# SimMAP Data Exploration

Notebook de consolidação dos dados do SimMAP a partir do CSV local do Forms e dos CSVs exportados do Notion. Ao final, o processo também atualiza `db/db.json` com `membros`, `projetos` e um bloco `analises` para reaproveitamento futuro.


In [1]:
from pathlib import Path
from difflib import get_close_matches
import json
import re

import pandas as pd
from IPython.display import display

from utils.sanitizer import (
    tratar_dataframe_notion,
    sanitize_string,
    build_nome_lookup,
    build_project_series,
    convert_comma_strings_to_lists,
    standardize_project_keys,
    split_participantes,
    normalizar_participante,
    merge_dataframe_columns_values,
    replace_nan_with_none,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


In [2]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

forms_path = Path("data/mapeamento-forms/Mapeamento de Membros LAWD (respostas).csv")
notion_dir = Path("data/notion-lawd")
projetos_dir = Path("data/notion-projetos")

negative_patterns = [
    "nao",
    "não",
    "nenhum",
    "nenhuma",
    "no momento nao",
    "no momento não",
    "nao possuo",
    "não possuo",
    "sem projeto",
    "nada no momento",
]

project_ignore_keys = {
    sanitize_string(value)
    for value in [
        "Nenhum",
        "Nenhum :(",
        "Buscando um projeto para atuar",
        "Nenhum :(, Buscando um projeto para atuar.",
    ]
}


def split_multi_value(value):
    if pd.isna(value):
        return []
    parts = [item.strip(" .") for item in re.split(r"[,;\n]+", str(value))]
    return [item for item in parts if item]


def clean_project_list(items):
    cleaned = []
    for item in items:
        key = sanitize_string(item)
        if pd.isna(key) or key in project_ignore_keys:
            continue
        cleaned.append(item)
    return list(dict.fromkeys(cleaned))


def extract_techs(value):
    if pd.isna(value):
        return []

    raw = str(value).replace("\r", "\n")
    chunks = []
    for piece in re.split(r"[\n;,]+", raw):
        piece = piece.strip()
        if not piece:
            continue
        if ":" in piece:
            piece = piece.split(":", 1)[1].strip()
        if not piece:
            continue
        if re.search(r"\be\b", piece, flags=re.IGNORECASE) and "," not in piece and "/" not in piece:
            subparts = [
                part.strip(" .")
                for part in re.split(r"\be\b", piece, flags=re.IGNORECASE)
                if part.strip(" .")
            ]
            chunks.extend(subparts)
            continue
        chunks.append(piece.strip(" ."))

    cleaned = []
    for item in chunks:
        item = re.sub(r"\s+", " ", item).strip(" .")
        if item:
            cleaned.append(item)
    return list(dict.fromkeys(cleaned))


def parse_score(value):
    if pd.isna(value):
        return pd.NA
    normalized = str(value).strip().replace(",", ".")
    return pd.to_numeric(normalized, errors="coerce")


def infer_positive_text(value):
    if pd.isna(value):
        return False
    normalized = sanitize_string(value)
    if pd.isna(normalized) or not normalized:
        return False
    normalized_lower = normalized.lower()
    return not any(pattern in normalized_lower for pattern in negative_patterns)


def count_list_items(series):
    exploded = series.explode().dropna().astype(str).str.strip()
    exploded = exploded[exploded != ""]
    return exploded.value_counts()


def value_counts_df(series, index_name, value_name="Quantidade"):
    counts = series.value_counts(dropna=False)
    return counts.rename_axis(index_name).reset_index(name=value_name)


def top_list_items_df(series, index_name, top_n=10):
    counts = count_list_items(series).head(top_n)
    return counts.rename_axis(index_name).reset_index(name="Quantidade")


def resolve_nome_canonico(nome, nome_lookup, canonical_names, fuzzy_cutoff=0.92):
    resolved = normalizar_participante(nome, nome_lookup)
    if resolved is not None:
        return resolved

    nome_sanitizado = sanitize_string(nome)
    if pd.isna(nome_sanitizado):
        return None

    matches = get_close_matches(nome_sanitizado, canonical_names, n=2, cutoff=fuzzy_cutoff)
    if len(matches) == 1:
        return matches[0]

    return None


def normalize_forms_dataframe(df_map_raw):
    rename_map = {
        "Carimbo de data/hora": "Data de Resposta",
        "Nome Completo: ": "Nome Completo",
        "Curso Atual": "Curso",
        "Período Atual:": "Período",
        "Há quanto tempo você faz parte da liga?": "Tempo de Liga",
        "Qual é a sua disponibilidade semanal voltada para atividades que envolvem a liga?": "Disponibilidade",
        "Qual objetivo pessoal impulsionou o seu interesse em se integrar à LAWD?": "Objetivo na LAWD",
        "Fale sobre as tecnologias que você possui conhecimento?": "Techs Declaradas",
        "Se precisasse pontuar seu conhecimento técnico de 0 a 10, qual seria a nota para as habilidades que você domina hoje?": "Nota Técnica",
        "Quais de seus atributos pessoais você considera mais relevantes para os projetos da LAWD?": "Atributos",
        "Qual perfil de especialidade em TI mais se alinha aos seus objetivos de carreira?": "Carreira",
        "Como você avalia sua maturidade na liga hoje?  ": "Maturidade",
        "Em quais projetos da LAWD você está inserido?": "Projetos Declarados no Forms",
        "Em quais projetos da LAWD você tem interesse em participar e contribuir? \n\n(Observação: Alguns projetos são novos e possuem um caráter amigável para iniciantes)": "Projetos com Interesse",
        "Existe algum projeto antigo da LAWD ou algum arquivo/iniciativa que você acredita que deveria ser retomado (por exemplo, o WebGarden)?\nSe sim, conte para nós qual projeto, por que ele merece ser retomado e como você imagina que isso poderia acontecer.": "Sugestão de Retomada",
        "Você possui algum projeto pessoal que gostaria de desenvolver dentro da LAWD?\nA ideia seria montar uma equipe para estruturar, validar e tirar esse projeto do papel.\n\nSe sim, conte um pouco sobre a proposta e como você imagina que ela poderia ser trabalhada.": "Proposta de Projeto Pessoal",
    }

    df_map = df_map_raw.rename(columns=rename_map).copy()
    df_map["Nome Completo Original"] = df_map["Nome Completo"]
    df_map["Nome Completo"] = df_map["Nome Completo"].apply(sanitize_string)
    df_map["Data de Resposta"] = pd.to_datetime(df_map["Data de Resposta"], dayfirst=True, errors="coerce")
    df_map["Período"] = pd.to_numeric(df_map["Período"], errors="coerce").astype("Int64")
    df_map["Nota Técnica"] = df_map["Nota Técnica"].apply(parse_score)

    for col in ["Atributos", "Carreira"]:
        df_map[col] = df_map[col].apply(split_multi_value)
    for col in ["Projetos Declarados no Forms", "Projetos com Interesse"]:
        df_map[col] = df_map[col].apply(lambda value: clean_project_list(split_multi_value(value)))

    df_map["Techs Declaradas"] = df_map["Techs Declaradas"].apply(extract_techs)
    df_map["Sugeriu Retomada"] = df_map["Sugestão de Retomada"].apply(infer_positive_text)
    df_map["Possui Projeto Pessoal"] = df_map["Proposta de Projeto Pessoal"].apply(infer_positive_text)

    ordered_columns = [
        "Nome Completo",
        "Nome Completo Original",
        "Data de Resposta",
        "Curso",
        "Período",
        "Tempo de Liga",
        "Disponibilidade",
        "Objetivo na LAWD",
        "Techs Declaradas",
        "Nota Técnica",
        "Atributos",
        "Carreira",
        "Maturidade",
        "Projetos Declarados no Forms",
        "Projetos com Interesse",
        "Sugestão de Retomada",
        "Sugeriu Retomada",
        "Proposta de Projeto Pessoal",
        "Possui Projeto Pessoal",
    ]
    return df_map[ordered_columns]


def ensure_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        return [value] if value.strip() else []
    return [value]


## Carregamento das bases

O pipeline usa três fontes locais: CSV do Forms, CSV geral de membros no Notion e CSV da base de projetos no Notion.


In [3]:
df_map_raw = pd.read_csv(forms_path)

df_notion1 = pd.read_csv(notion_dir / "Membros b8415a3d3d19462690637c8ea91a09e3_all.csv")
df_notion2 = pd.read_csv(notion_dir / "Membros b8415a3d3d19462690637c8ea91a09e3.csv")
df_notion = pd.merge(df_notion1, df_notion2, on="Name")
df_notion = tratar_dataframe_notion(
    df_notion,
    colunas_para_remover=["Resumo da IA", "Seleção", "Arquivos e mídia", "Cadastro"],
    colunas_com_links=["Projetos"],
)

df_projetos1 = pd.read_csv(projetos_dir / "Projetos 1dfb762c7e8b80fa84dfc9ad8e619d61_all.csv")
df_projetos2 = pd.read_csv(projetos_dir / "Projetos 1dfb762c7e8b80fa84dfc9ad8e619d61.csv")
df_projetos = pd.merge(df_projetos1, df_projetos2, on="Nome do Projeto")
df_projetos = tratar_dataframe_notion(
    df_projetos,
    colunas_para_remover=["Documentação", "Link para o repositório", "Progresso"],
    colunas_com_links=df_projetos.select_dtypes(include=["object", "string"]).columns,
)

df_forms = normalize_forms_dataframe(df_map_raw)

display(pd.DataFrame({
    "Base": ["Forms", "Notion membros", "Notion projetos"],
    "Linhas": [len(df_forms), len(df_notion), len(df_projetos)],
}))
display(df_forms.head(3))


,Base,Linhas
0,Forms,31
1,Notion membros,33
2,Notion projetos,17


,Nome Completo,Nome Completo Original,Data de Resposta,Curso,Período,Tempo de Liga,Disponibilidade,Objetivo na LAWD,Techs Declaradas,Nota Técnica,Atributos,Carreira,Maturidade,Projetos Declarados no Forms,Projetos com Interesse,Sugestão de Retomada,Sugeriu Retomada,Proposta de Projeto Pessoal,Possui Projeto Pessoal
0,BRUNO SAINT CLAIR SILVA OLIVEIRA,Bruno Saint Clair Silva Oliveira,2026-03-03 14:41:05,SI (Sistemas de Informação),8,1 ano,De 6 a 8 horas semanais,"Desenvolvimento pessoal, tanto em relação a conhecimento técnico quanto a colaboração, trabalho em equipe e tudo mais.","[Java, JavaScript, TypeScript, Python, HTML, CSS, SpringBoot, Express, FastApi, React, TailwindCSS, um pouquinho de ...",7.5,"[Autonomia para aprender, Comunicação, Trabalho em equipe]",[Desenvolvedor Back-end],Autônomo (Resolve problemas sozinho),[SIMGrade],"[Site da LAWD, Projeto Oportunidade Aprendiz (POA/MPSE)]",Site do Dcomp.,True,No momento não.,False
1,GABRIEL ARGOLO,Gabriel Argôlo,2026-03-03 14:08:06,CC (Ciência da Computação),10,2 anos.,De 6 a 8 horas semanais,"Aplicar o conhecimento obtido na academia para resolver problemas reais, sejam eles, da própria academia ou não.","[C#, Java]",8.0,"[Autonomia para aprender, Comunicação, Organização, Proatividade, Trabalho em equipe]",[Desenvolvedor Back-end],Autônomo (Resolve problemas sozinho),[ufs.br],[ufs.br],"Acho que a liga está com muitos projetos mais urgentes no momento. Mas, futuramente eu apoio a retomada do WebGarden.",True,Não possuo.,False
2,IRLAN FELIPE ALVES DOS SANTOS,Irlan Felipe Alves dos Santos,2026-03-02 10:28:52,EC (Engenharia da Computação),4,1 ano,De 9 a 12 horas semanais,Eu tinha interesse em estar com pessoas que estivessem envolvidas com projetos web e também queria desenvolver minha...,"[Vanilla, conhecimento básico em react, node]",6.0,"[Autonomia para aprender, Comunicação]",[Desenvolvedor Back-end],Em desenvolvimento (Já executa tarefas com apoio),[],"[Site da LAWD, SIMCalc (módulo de calculadora de horas complementares do SIMGrade)]",Acredito que nenhum projeto.,False,NaN,False


## Consolidação dos membros

Os nomes do Forms são alinhados à chave canônica do Notion antes do merge. Para casos com pequenas divergências ortográficas, há uma etapa de aproximação controlada.


In [4]:
df_notion = df_notion.drop(columns=["Name", "Phone", "Email"])
df_notion["Nome Completo Original"] = df_notion["Nome Completo"]
df_notion["Nome Completo"] = df_notion["Nome Completo"].apply(sanitize_string)

nome_lookup_forms = build_nome_lookup(df_notion)
canonical_names = sorted(df_notion["Nome Completo"].dropna().unique())

df_forms["Nome Forms Sanitizado"] = df_forms["Nome Completo"]
df_forms["Nome Completo"] = df_forms["Nome Completo"].apply(
    lambda nome: resolve_nome_canonico(nome, nome_lookup_forms, canonical_names) or nome
)

df = pd.merge(df_notion, df_forms, on="Nome Completo", how="left", suffixes=("_notion", "_forms"))
df = df.rename(columns={"Nome Completo Original_notion": "Nome Completo Original"})
df = df.drop(columns=["Nome Completo Original_forms", "Projetos"], errors="ignore")

time_col = "times" if "times" in df_projetos.columns else "Time"
nome_lookup = build_nome_lookup(df)

df_projetos[time_col] = df_projetos[time_col].apply(
    lambda value: [
        nome
        for nome in (
            resolve_nome_canonico(item, nome_lookup, canonical_names)
            for item in split_participantes(value)
        )
        if nome is not None
    ]
)

df_projetos["Responsável"] = df_projetos["Responsável"].apply(
    lambda value: ", ".join(
        [
            nome
            for nome in (
                resolve_nome_canonico(item, nome_lookup, canonical_names)
                for item in split_participantes(value)
            )
            if nome is not None
        ]
    ) if not pd.isna(value) else value
)

series_projetos = build_project_series(
    df_projetos=df_projetos,
    nome_lookup=nome_lookup,
    project_col="Nome do Projeto",
    time_col=time_col,
    responsavel_col="Responsável",
    status_col="Status",
)

for base_col in series_projetos.keys():
    for suffix in ["", "_x", "_y"]:
        df = df.drop(columns=[f"{base_col}{suffix}"], errors="ignore")

for serie in series_projetos.values():
    df = df.merge(serie, on="Nome Completo", how="left")

for col in [
    "Projetos Atuais",
    "Projetos Atuais Em Andamento",
    "Projetos Finalizados",
    "Projetos Coordenados",
    "Projetos Declarados no Forms",
    "Projetos com Interesse",
    "Atributos",
    "Carreira",
    "Techs Declaradas",
]:
    if col not in df.columns:
        df[col] = [[] for _ in range(len(df))]

df = convert_comma_strings_to_lists(df)
df_projetos = convert_comma_strings_to_lists(df_projetos)

project_aliases = {
    "ufs.br": "UFS.BR",
    "Projeto Oportunidade Aprendiz (POA/MPSE)": "POA - MPSE",
    "Projeto oportunidade aprendiz (POA/MPSE)": "POA - MPSE",
    "Todo o nivelamento para o desenvolvimento de projetos": "CriaJunto",
    "Cria Junto": "CriaJunto",
    "Cria Junto (Mini POC do nivelamento)": "CriaJunto",
    "SIMCalc (módulo de calculadora de horas complementares do SIMGrade)": "SIMCalc",
    "TransformAção (Projeto social em parceria com a LADATA)": "TransformAção",
    "Site do Dcomp": "Site do DComp",
    '"Atualização" do Site do PS': "Site do PS 2.0",
}

df_projetos, df = standardize_project_keys(
    df_projetos=df_projetos,
    df=df,
    project_col="Nome do Projeto",
    df_project_columns=[
        "Projetos Declarados no Forms",
        "Projetos Atuais",
        "Projetos Atuais Em Andamento",
        "Projetos Finalizados",
        "Projetos Coordenados",
        "Projetos com Interesse",
    ],
    aliases=project_aliases,
    report_unmatched=True,
)

df = merge_dataframe_columns_values(
    df=df,
    left_col="Techs",
    right_col="Techs Declaradas",
    target_col="Techs",
    drop_source_columns=True,
)

df["Projetos com Interesse"] = df.apply(
    lambda row: [
        projeto
        for projeto in (row["Projetos com Interesse"] if isinstance(row["Projetos com Interesse"], list) else [])
        if projeto not in (
            row["Projetos Atuais Em Andamento"]
            if isinstance(row["Projetos Atuais Em Andamento"], list)
            else []
        )
    ],
    axis=1,
)

df["Período"] = pd.to_numeric(df["Período"], errors="coerce").astype("Int64")
df["Data de Resposta"] = df["Data de Resposta"].dt.strftime("%Y-%m-%d %H:%M:%S").where(df["Data de Resposta"].notna())

for col in [
    "Áreas de atuação",
    "Áreas de interesse",
    "Techs",
    "Atributos",
    "Carreira",
    "Projetos Declarados no Forms",
    "Projetos Atuais Em Andamento",
    "Projetos Finalizados",
    "Projetos Coordenados",
    "Projetos com Interesse",
]:
    if col in df.columns:
        df[col] = df[col].apply(ensure_list)

member_columns = [
    "Nome Completo",
    "Áreas de atuação",
    "Projetos Atuais Em Andamento",
    "Projetos Finalizados",
    "Projetos Coordenados",
    "Projetos com Interesse",
    "Projetos Declarados no Forms",
    "Techs",
    "Período",
    "Áreas de interesse",
    "Disponibilidade",
    "Carreira",
    "Maturidade",
    "Curso",
    "Entrada",
    "Tempo de Liga",
    "Data de Resposta",
    "Nota Técnica",
    "Atributos",
    "Objetivo na LAWD",
    "Sugestão de Retomada",
    "Sugeriu Retomada",
    "Proposta de Projeto Pessoal",
    "Possui Projeto Pessoal",
    "Nome Completo Original",
    "Aniversário",
    "Status",
]

rest = [
    col for col in df.columns
    if col not in member_columns and col not in ["Nome Forms Sanitizado", "Projetos Atuais"]
]
df = df[member_columns + rest]

project_columns = [
    "Nome do Projeto",
    "Nome do Projeto Original",
    "Responsável",
    "Data de Início",
    "Data de Encerramento",
    "Status",
    "Prioridade",
    "Time",
]
df_projetos = df_projetos[[col for col in project_columns if col in df_projetos.columns]]

display(df.head(5))
display(df_projetos.head(5))


Projetos sem correspondencia com o padrao de df_projetos:
['Dcomp Mulheres']


,Nome Completo,Áreas de atuação,Projetos Atuais Em Andamento,Projetos Finalizados,Projetos Coordenados,Projetos com Interesse,Projetos Declarados no Forms,Techs,Período,Áreas de interesse,Disponibilidade,Carreira,Maturidade,Curso,Entrada,Tempo de Liga,Data de Resposta,Nota Técnica,Atributos,Objetivo na LAWD,Sugestão de Retomada,Sugeriu Retomada,Proposta de Projeto Pessoal,Possui Projeto Pessoal,Nome Completo Original,Aniversário,Status
0,GYOVANI YURI SOUZA SANTOS,[Eventos],"[SIMGRADE, UFS.BR]",[FISHBASE],"[SIMGRADE, UFS.BR]",[SITE DA LAWD],"[UFS.BR, SIMGRADE]","[CAD, Redes, TS, React, Tailwind, Next, Nest, JS, Python, FastAPI, OpenCV, SonarQube, outros]",8,"[Desenvolvimento web, Redes]",De 6 a 8 horas semanais,"[Full Stack, Gestor de Projeto / Scrum]",Mentor (Ajuda outros membros),SI (Sistemas de Informação),NaN,3 anos,2026-03-03 14:05:06,9.0,"[Autonomia para aprender, Trabalho em equipe]",Melhorar o DCOMP,Não,False,NaN,False,Gyovani Yuri Souza Santos,19 de março de 2025,membro efetivo
1,JESSICA CARVALHO VIANA,[Ensino],[],[],[],[UFS.BR],[UFS.BR],"[Backend, Design, FrontEnd, Frontend, HTML/css, React, flutter]",7,[Eventos],Menos de 6 horas semanais,"[Desenvolvedor Front-end, UI/UX, Gestor de Projeto / Scrum, Pesquisador / Documentação]",Autônomo (Resolve problemas sozinho),Design,1 de fevereiro de 2024,2 anos,2026-03-05 19:09:49,8.0,"[Autonomia para aprender, Comunicação, Liderança, Organização, Proatividade, Trabalho em equipe]","[Fazer parte de um time empolgado e evoluir em conjunto, atuando em projetos que trazem impacto a comunidade]","[Webgarden, acredito que com o avanço do UFS.br esse projeto pode ser integrado um dia]",True,NaN,False,Jéssica Carvalho Viana,7 de agosto de 2025,membro efetivo
2,MARIA LUIZA FERREIRA MATOS,[Projetos],[UFS.BR],[],[],[SIMCALC],[UFS.BR],"[Backend, Design, FrontEnd, HTML, CSS, JS, TS, React]",8,"[Marketing, Pesquisa, Projetos]",De 9 a 12 horas semanais,"[Desenvolvedor Front-end, UI/UX, Gestor de Projeto / Scrum]",Autônomo (Resolve problemas sozinho),CC (Ciência da Computação),5 de fevereiro de 2024,2 anos,2026-03-03 14:14:32,7.0,"[Autonomia para aprender, Liderança, Organização, Trabalho em equipe]","[Meu objetivo era sair um pouco da bolha da sala de aula. Sinto que a faculdade às vezes fica muito na teoria, e eu ...",não lembro 😭😭,False,NaN,False,Maria Luiza Ferreira Matos,22 de julho de 2025,membro efetivo
3,JOAO FELIPE QUENTINO,[Ensino],"[POA - MPSE, UFS.BR]",[CHATBOT],"[CHATBOT, POA - MPSE]",[TRANSFORMACAO],[POA - MPSE],"[Automação, IA, Machine Learning, Python, python, data science, web, backend]",9,"[IA, Pesquisa]",Menos de 6 horas semanais,"[Desenvolvedor Back-end, Pesquisador / Documentação]",Mentor (Ajuda outros membros),SI (Sistemas de Informação),NaN,Desde da fundação,2026-03-05 23:46:49,9.0,"[Autonomia para aprender, Comunicação, Liderança, Organização, Proatividade, Trabalho em equipe]",mudar a faculdade,nao existe,False,NaN,False,João Felipe Quentino,6 de abril de 2025,membro efetivo
4,WANESSA SILVA SANTOS,[RH],[],[],[],[SITE DA LAWD],[DCOMP MULHERES],"[Adm, Java, Vanilla, Python, Javascript, C]",8,"[Projetos, RH]",De 6 a 8 horas semanais,"[Desenvolvedor Front-end, Gestor de Projeto / Scrum]",Autônomo (Resolve problemas sozinho),SI (Sistemas de Informação),NaN,Aproximadamente 2 anos,2026-03-06 16:48:08,NaN,"[Comunicação, Organização, Proatividade]",Vontade de aprender novas ferramentas de trabalho e impulsionar habilidades que gostaria de exercitar,"[Projeto do Calicomp, tiveram alguns impecilhos da parte externa que não envolvia a LAWD, mas acho que é muito impor...",False,NaN,False,Wanessa Silva Santos,27 de abril de 2025,diretor


,Nome do Projeto,Nome do Projeto Original,Responsável,Data de Início,Data de Encerramento,Status,Prioridade,Time
0,FISHBASE,Fishbase,DANIEL JOSE SILVA TRINDADE,01/04/2025,09/06/2025,Finalizado,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, DAVI BITTENCOURT DE ALMEIDA, GYOVANI YURI SOUZA SANTOS]"
1,UFS.BR,UFS.BR,GYOVANI YURI SOUZA SANTOS,09/06/2025,01/12/2026,Em andamento,Alta,"[DAVI BITTENCOURT DE ALMEIDA, GABRIEL ARGOLO JULIAO DOS SANTOS, GUSTAVO HENRIQUE ARAGAO SILVA, GYOVANI YURI SOUZA SA..."
2,SITE DO PS,Site do PS,DAVI BITTENCOURT DE ALMEIDA,08/05/2024,15/03/2025,Finalizado,Não se aplica,"[DAVI BITTENCOURT DE ALMEIDA, GUILHERME LINARD PEREIRA, VINICIUS MORAIS SOUZA, ALVARO LUIS SILVA PEIXOTO]"
3,CHATBOT,Chatbot,JOAO FELIPE QUENTINO,15/04/2024,03/02/2025,Finalizado,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, JOAO FELIPE QUENTINO, ALVARO LUIS SILVA PEIXOTO]"
4,WEBGARDEN,WebGarden,,24/01/2025,01/08/2025,Suspenso,Não se aplica,[]


## Cobertura e consistência

Aqui ficam os indicadores de cobertura entre Forms e Notion, além dos nomes que ainda permanecem sem correspondência exata.


In [5]:
forms_without_notion = df_forms.loc[
    ~df_forms["Nome Completo"].isin(set(df_notion["Nome Completo"])),
    ["Nome Completo Original"],
].drop_duplicates()

notion_without_forms = df.loc[
    df["Data de Resposta"].isna(),
    ["Nome Completo Original"],
].drop_duplicates()

stats_resumo = pd.DataFrame([
    {"Métrica": "Respostas no Forms", "Valor": int(len(df_forms))},
    {"Métrica": "Membros no Notion", "Valor": int(len(df_notion))},
    {"Métrica": "Membros do Notion com resposta no Forms", "Valor": int(df["Data de Resposta"].notna().sum())},
    {"Métrica": "Respostas do Forms sem match no Notion", "Valor": int(len(forms_without_notion))},
    {"Métrica": "Membros do Notion sem resposta no Forms", "Valor": int(len(notion_without_forms))},
    {"Métrica": "Média da nota técnica", "Valor": round(float(df_forms["Nota Técnica"].dropna().mean()), 2)},
    {"Métrica": "Membros com sugestão de retomada", "Valor": int(df_forms["Sugeriu Retomada"].sum())},
    {"Métrica": "Membros com projeto pessoal declarado", "Valor": int(df_forms["Possui Projeto Pessoal"].sum())},
])

display(stats_resumo)
display(forms_without_notion.rename(columns={"Nome Completo Original": "Respostas sem match no Notion"}))
display(notion_without_forms.rename(columns={"Nome Completo Original": "Membros do Notion sem resposta no Forms"}))


,Métrica,Valor
0,Respostas no Forms,31.00
1,Membros no Notion,33.00
2,Membros do Notion com resposta no Forms,30.00
3,Respostas do Forms sem match no Notion,1.00
4,Membros do Notion sem resposta no Forms,3.00
5,Média da nota técnica,7.12
6,Membros com sugestão de retomada,11.00
7,Membros com projeto pessoal declarado,4.00


,Respostas sem match no Notion
8,Diogo Eduardo Alves Martins


,Membros do Notion sem resposta no Forms
5,Alipio Fenando de Paula Pires
20,Paulo Marques de Oliveira Silva
29,Guilherme Lavrador Viana


## Perfil geral dos respondentes

As tabelas abaixo resumem curso, disponibilidade, maturidade e a distribuição da nota técnica por estágio dentro da liga.


In [6]:
stats_por_curso = value_counts_df(df_forms["Curso"], "Curso")
stats_por_disponibilidade = value_counts_df(df_forms["Disponibilidade"], "Disponibilidade")
stats_por_maturidade = value_counts_df(df_forms["Maturidade"], "Maturidade")

nota_por_maturidade = (
    df_forms.dropna(subset=["Nota Técnica"])
    .groupby("Maturidade", dropna=False)["Nota Técnica"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "Respondentes", "mean": "Nota Média"})
)
nota_por_maturidade["Nota Média"] = nota_por_maturidade["Nota Média"].round(2)

display(stats_por_curso)
display(stats_por_disponibilidade)
display(stats_por_maturidade)
display(nota_por_maturidade)


,Curso,Quantidade
0,SI (Sistemas de Informação),15
1,CC (Ciência da Computação),9
2,EC (Engenharia da Computação),3
3,Design,3
4,Formado,1


,Disponibilidade,Quantidade
0,De 6 a 8 horas semanais,19
1,Menos de 6 horas semanais,8
2,De 9 a 12 horas semanais,4


,Maturidade,Quantidade
0,Autônomo (Resolve problemas sozinho),11
1,Em desenvolvimento (Já executa tarefas com apoio),11
2,Mentor (Ajuda outros membros),8
3,Iniciante (Ainda aprendendo fundamentos),1


,Maturidade,Respondentes,Nota Média
0,Autônomo (Resolve problemas sozinho),9,7.22
1,Em desenvolvimento (Já executa tarefas com apoio),7,6.43
2,Iniciante (Ainda aprendendo fundamentos),1,6.00
3,Mentor (Ajuda outros membros),7,7.86


## Interesses e sinais úteis para análise

O Forms traz campos antes descartados que ajudam a inferir afinidades técnicas, interesses de carreira e disposição para novas frentes da liga.


In [7]:
stats_por_carreira = top_list_items_df(df_forms["Carreira"], "Carreira", top_n=15)
stats_por_atributo = top_list_items_df(df_forms["Atributos"], "Atributo", top_n=15)
stats_por_tech = top_list_items_df(df_forms["Techs Declaradas"], "Tecnologia", top_n=20)
stats_por_projeto_interesse = top_list_items_df(df_forms["Projetos com Interesse"], "Projeto", top_n=20)

stats_extra = pd.DataFrame([
    {"Indicador": "Sugestões de retomada", "Quantidade": int(df_forms["Sugeriu Retomada"].sum())},
    {"Indicador": "Projetos pessoais declarados", "Quantidade": int(df_forms["Possui Projeto Pessoal"].sum())},
])

display(stats_por_carreira)
display(stats_por_atributo)
display(stats_por_tech)
display(stats_por_projeto_interesse)
display(stats_extra)


,Carreira,Quantidade
0,Full Stack,15
1,Desenvolvedor Back-end,12
2,Gestor de Projeto / Scrum,11
3,Desenvolvedor Front-end,11
4,UI/UX,9
5,DevOps,5
6,Pesquisador / Documentação,4


,Atributo,Quantidade
0,Trabalho em equipe,27
1,Autonomia para aprender,23
2,Comunicação,22
3,Organização,19
4,Proatividade,15
5,Liderança,8


,Tecnologia,Quantidade
0,CSS,11
1,React,11
2,Java,10
3,Python,10
4,HTML,8
5,JS,8
6,TypeScript,6
7,C,5
8,JavaScript,4
9,PostgreSQL,4


,Projeto,Quantidade
0,Site da LAWD,19
1,ufs.br,13
2,SIMCalc (módulo de calculadora de horas complementares do SIMGrade),10
3,Projeto Oportunidade Aprendiz (POA/MPSE),9
4,TransformAção (Projeto social em parceria com a LADATA),8
5,Landing Page do WWWeek,7


,Indicador,Quantidade
0,Sugestões de retomada,11
1,Projetos pessoais declarados,4


## Exportação

Além dos registros de `membros` e `projetos`, o JSON final passa a carregar um bloco `analises` com os principais agregados calculados neste notebook.


In [8]:
analysis_payload = {
    "resumo": stats_resumo.to_dict(orient="records"),
    "porCurso": stats_por_curso.to_dict(orient="records"),
    "porDisponibilidade": stats_por_disponibilidade.to_dict(orient="records"),
    "porMaturidade": stats_por_maturidade.to_dict(orient="records"),
    "carreirasMaisCitadas": stats_por_carreira.to_dict(orient="records"),
    "atributosMaisCitados": stats_por_atributo.to_dict(orient="records"),
    "techsMaisCitadas": stats_por_tech.to_dict(orient="records"),
    "projetosDeInteresse": stats_por_projeto_interesse.to_dict(orient="records"),
    "notaTecnicaPorMaturidade": nota_por_maturidade.to_dict(orient="records"),
    "nomesSemMatch": {
        "formsSemNotion": forms_without_notion["Nome Completo Original"].dropna().tolist(),
        "notionSemForms": notion_without_forms["Nome Completo Original"].dropna().tolist(),
    },
}

db = {
    "membros": df.to_dict(orient="records"),
    "projetos": df_projetos.to_dict(orient="records"),
    "analises": analysis_payload,
}

db = replace_nan_with_none(db)
output_path = project_root / "db" / "db.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"db.json salvo em: {output_path}")


db.json salvo em: /home/gustavo/Documents/lawd/gestao/SimMAP/db/db.json
